# STEP 1. Load + unify schema + range filter + completeness checks

## What changed in this revision

| | Before | Now |
|---|---|---|
| Total store count | 2023·24 used `nonfranchise_store_count` (= non-franchise only), 2025~ used `store_count` → **definition discontinuity** | Both unified to `store_count` (total) |
| `franchise_store_count` | unused | kept (a franchise share can be derived) |
| Filter timing | 100 service categories carried through to STEP 5 | **Reduced to 10 service categories · 12 quarters in STEP 1** |
| Completeness checks | none | cross-tab + quarter-continuity diagnostics added |
| Raw data | overwritten | `_raw` preserved, work on a `.copy()` |

## Why the store-count mapping was changed

In the two generations of raw files, the following identities hold in **100%** of rows.

```
2023·2024 :  similar-category store count = nonfranchise_store_count + franchise_store_count
2025~     :  store_count                    = nonfranchise_store_count + franchise_store_count
```

That is, the old files' `nonfranchise_store_count` is not the total but the **non-franchise store count**. The earlier code put the old files' partial count and the new files' total `store_count` into the same column, so the mean **jumped artificially by +18%** between 2024Q4 → 2025Q1.

The degree of undercounting differs by service category: fried-chicken 64%, fast-food 50%, Korean restaurants 10%. The treated group was barely cut while franchise-heavy categories were cut sharply, so **the matching covariates themselves were distorted**.

## Analysis scope

- **12 quarters** 20231 ~ 20261
- **10 food-service categories**, specified by code (`CS100001`~`CS100010`)

> ⚠️ The label `y(t)` looks at quarter `t+1`. Since 20261 is not used, **the span that gets labels is 20231~20261 (12 quarters)**. Rows for 20254 are dropped in STEP 3 for having no label.

In [1]:
import sys, os
from pathlib import Path

# Wherever the notebook is opened, find the project root and add it to sys.path
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("Project root:", ROOT.name)

Project root: seoul-closure-risk


In [2]:
from config import (RAW, PROC, ENCODING, FILES, QUARTERS, SVC_CODES, FOOD,
                    STORE_RENAME_OLD, STORE_KEEP, SALES_KEEP, TARGET_TYPE, standardize_source)

CODES = list(SVC_CODES.keys())
print(f"Analysis quarters {len(QUARTERS)} : {QUARTERS[0]} ~ {QUARTERS[-1]}")
print(f"Analysis service categories {len(CODES)} : {', '.join(FOOD)}")

def load(key):
    p = RAW / FILES[key]
    df = standardize_source(pd.read_csv(p, encoding=ENCODING, low_memory=False))
    print(f"  {key:12} {df.shape[0]:>8,} rows × {df.shape[1]:>2} cols")
    return df

Analysis quarters 13 : 20231 ~ 20261
Analysis service categories 10 : Korean restaurants, Chinese restaurants, Japanese restaurants, Western restaurants, Bakeries, Fast-food restaurants, Fried-chicken shops, Korean snack-food restaurants, Beer and casual pubs, Coffee and beverages


## 1-1. Load the raw data (preserve `_raw`)

Never modify `_raw`. If something goes wrong midway, come back to it.

In [3]:
print("[stores]")
store23_raw = load("store_2023")
store24_raw = load("store_2024")
store25_raw = load("store_2025")

print("[sales]")
sales23_raw   = load("sales_2023")
sales24_raw   = load("sales_2024")
sales25_raw   = load("sales_2025")
sales_late_raw = load("sales_late")

print("[commercial-area attributes]")
area_raw     = load("area")
flow_raw     = load("flow")
facility_raw = load("facility")

[stores]


  store_2023    307,741 rows × 11 cols


  store_2024    306,889 rows × 11 cols


  store_2025    380,747 rows × 11 cols
[sales]


  sales_2023     88,246 rows ×  8 cols


  sales_2024     87,179 rows ×  8 cols


  sales_2025     85,732 rows ×  8 cols


  sales_late    106,920 rows ×  8 cols
[commercial-area attributes]
  area            1,650 rows ×  9 cols
  flow           34,633 rows × 13 cols


  facility       33,138 rows ×  6 cols


## 1-2. Store schema diagnostics

Check by eye what differs **before** renaming.

In [4]:
c23, c24, c25 = (set(d.columns) for d in (store23_raw, store24_raw, store25_raw))
print("2023 vs 2024      :", sorted(c23 ^ c24) or "no difference")
print("2024 vs 2025~     :", sorted(c24 ^ c25))
print()
print("[Identity check] verify arithmetically that the two generations share the same structure")
for nm, d, tot, gen in [("2023 ", store23_raw, "legacy_total_store_count", "legacy_nonfranchise_store_count"),
                        ("2024 ", store24_raw, "legacy_total_store_count", "legacy_nonfranchise_store_count"),
                        ("2025~", store25_raw, "store_count",     "nonfranchise_store_count")]:
    hit = (d[tot] == d[gen] + d["franchise_store_count"]).mean()
    print(f"  {nm}  {tot:14} == {gen:12} + franchise_store_count   match rate {hit:.4f}")
print()
print("→ In the old files, the 'store count' column is non-franchise, not the total. The 'similar-category store count' is the total.")

2023 vs 2024      : no difference
2024 vs 2025~     : ['legacy_nonfranchise_store_count', 'legacy_total_store_count', 'nonfranchise_store_count', 'store_count']

[Identity check] verify arithmetically that the two generations share the same structure
  2023   legacy_total_store_count == legacy_nonfranchise_store_count + franchise_store_count   match rate 1.0000
  2024   legacy_total_store_count == legacy_nonfranchise_store_count + franchise_store_count   match rate 1.0000
  2025~  store_count    == nonfranchise_store_count + franchise_store_count   match rate 1.0000

→ In the old files, the 'store count' column is non-franchise, not the total. The 'similar-category store count' is the total.


## 1-3. Unify the schema — to the new file's names

Make a working copy with `.copy()` and align the old files' column names to the new file.

In [5]:
store23 = store23_raw.copy().rename(columns=STORE_RENAME_OLD)
store24 = store24_raw.copy().rename(columns=STORE_RENAME_OLD)
store25 = store25_raw.copy()

c23, c24, c25 = (set(d.columns) for d in (store23, store24, store25))
assert not (c23 ^ c24) and not (c24 ^ c25), "Columns still mismatch after renaming"
print("Column difference after renaming : none ✅")

store = pd.concat([store23, store24, store25], axis=0, ignore_index=True)
assert len(store) == len(store23) + len(store24) + len(store25), "Rows lost during concat"
print(f"Merged {len(store):,} rows  (= {len(store23):,} + {len(store24):,} + {len(store25):,}) ✅")
print(f"Duplicate rows {store.duplicated().sum():,}")
print(f"Raw quarters {sorted(store['quarter_code'].unique())}")
print("  → 20261 is cut off in the next cell (analysis scope is 12 quarters)")

Column difference after renaming : none ✅
Merged 995,377 rows  (= 307,741 + 306,889 + 380,747) ✅


Duplicate rows 0
Raw quarters [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]
  → 20261 is cut off in the next cell (analysis scope is 12 quarters)


### Definition-continuity check

Check that this fix actually removed the discontinuity. Quarterly mean store count for food-service categories in neighborhood commercial areas.

`store_count` should connect smoothly from 2024Q4 → 2025Q1, and so should `nonfranchise_store_count`. The values the earlier code produced (non-franchise for old files, total for new files) show a jump here.

In [6]:
chk = store[(store["area_type_name"] == TARGET_TYPE) &
            (store["service_code"].isin(CODES))]
t = chk.groupby("quarter_code")[["store_count", "nonfranchise_store_count"]].mean()
t["previous_code_value"] = np.where(t.index >= 20251, t["store_count"], t["nonfranchise_store_count"])
display(t.round(2))
print("Note: 'store_count' is continuous from 20244 (6.23) → 20251 (6.16).")
print("   'previous_code_value' jumps +18% from 5.22 → 6.16 — this is the problem that was fixed.")
print("   (The 20261 row is for reference only. It is not used in the analysis.)")

,store_count,nonfranchise_store_count,previous_code_value
quarter_code,,,
20231,6.23,5.21,5.21
20232,6.24,5.22,5.22
20233,6.24,5.22,5.22
20234,6.26,5.23,5.23
20241,6.29,5.26,5.26
20242,6.29,5.27,5.27
20243,6.25,5.24,5.24
20244,6.23,5.22,5.22
20251,6.16,5.17,6.16


Note: 'store_count' is continuous from 20244 (6.23) → 20251 (6.16).
   'previous_code_value' jumps +18% from 5.22 → 6.16 — this is the problem that was fixed.
   (The 20261 row is for reference only. It is not used in the analysis.)


## 1-4. Compute commercial-area denominators — **before the service-category filter**

Once the service categories are cut to 10, `area_total_stores` (sum over all 100 categories) can no longer be built. The denominator must be computed now.

In [7]:
store = store[store["quarter_code"].isin(QUARTERS)].copy()
print(f"After 12-quarter filter {len(store):,} rows ({store['service_code'].nunique()} service categories)")

key = ["quarter_code", "area_code"]
all_stores = store.groupby(key)["store_count"].sum().rename("area_total_stores")
food_stores = (store[store["service_code"].isin(CODES)]
        .groupby(key)["store_count"].sum().rename("area_food_stores"))

denom = pd.concat([all_stores, food_stores], axis=1).fillna({"area_food_stores": 0}).reset_index()
denom["food_service_share"] = denom["area_food_stores"] / denom["area_total_stores"].replace(0, np.nan)
print(f"Denominator table {denom.shape}")
display(denom[["area_total_stores", "area_food_stores", "food_service_share"]].describe().round(3))

After 12-quarter filter 995,377 rows (100 service categories)


Denominator table (21450, 5)


,area_total_stores,area_food_stores,food_service_share
count,21450.000,21450.000,21450.000
mean,319.382,81.098,0.265
std,673.082,140.279,0.116
min,1.000,0.000,0.000
25%,72.000,16.000,0.193
50%,155.000,39.000,0.257
75%,297.000,84.000,0.328
max,15131.000,1724.000,0.747


## 1-5. Service-category and column filter

This shrinks the data to 1/10. Every later step becomes lighter.

In [8]:
before = len(store)
store = store[store["service_code"].isin(CODES)][STORE_KEEP].copy()
store = store.merge(denom, on=key, how="left", validate="many_to_one")

store["franchise_share"] = store["franchise_store_count"] / store["store_count"].replace(0, np.nan)

print(f"Service-category filter {before:,} → {len(store):,} rows  ({len(store)/before:.1%})")
print(f"Columns {len(store.columns)}")
display(store.head(3))

Service-category filter 995,377 → 160,352 rows  (16.1%)
Columns 14


,quarter_code,area_type_name,area_code,service_code,service_name,store_count,nonfranchise_store_count,franchise_store_count,opening_count,closure_count,area_total_stores,area_food_stores,food_service_share,franchise_share
0,20231,neighborhood commercial area,3110001,CS100001,Korean restaurants,11,10,1,1,0,49,19.0,0.387755,0.090909
1,20231,neighborhood commercial area,3110001,CS100003,Japanese restaurants,1,1,0,0,0,49,19.0,0.387755,0.000000
2,20231,neighborhood commercial area,3110001,CS100008,Korean snack-food restaurants,3,3,0,0,0,49,19.0,0.387755,0.000000


## 1-6. Completeness check ①  Cross-tab

For district-level data this table would be full at 25 (the number of districts). **Commercial-area level is different** — the number of commercial areas present differs by service category and fluctuates by quarter. That fluctuation is what makes the cells for which STEP 3 cannot build a label.

In [9]:
gm = store[store["area_type_name"] == TARGET_TYPE]
ct = pd.crosstab(gm["quarter_code"], gm["service_name"], margins=True)
ct = ct[[c for c in FOOD if c in ct.columns] + ["All"]]
display(ct)

n_area = gm["area_code"].nunique()
print(f"{TARGET_TYPE} commercial areas {n_area} × 13 quarters = {n_area*13:,}  (full-occupancy baseline per service category)")
cov = (gm.groupby("service_name").size() / (n_area * 13)).sort_values(ascending=False)
display(cov.mul(100).round(1).rename("coverage_pct").to_frame())
print("Note: the lower a service category's coverage, the narrower the pool of commercial areas usable as controls.")

service_name,Korean restaurants,Chinese restaurants,Japanese restaurants,Western restaurants,Bakeries,Fast-food restaurants,Fried-chicken shops,Korean snack-food restaurants,Beer and casual pubs,Coffee and beverages,All
quarter_code,,,,,,,,,,,
20231,1047,708,609,663,744,721,746,855,850,1015,7958
20232,1043,710,611,666,748,718,747,849,849,1019,7960
20233,1042,710,608,659,749,717,742,850,850,1019,7946
20234,1040,705,607,656,752,718,747,851,855,1023,7954
20241,1041,703,614,658,754,716,747,858,853,1024,7968
20242,1042,697,609,661,753,705,740,859,850,1023,7939
20243,1043,693,604,656,749,699,735,859,852,1025,7915
20244,1044,693,605,647,751,693,733,858,851,1027,7902
20251,1044,690,601,651,753,692,726,856,848,1025,7886


neighborhood commercial area commercial areas 1084 × 13 quarters = 14,092  (full-occupancy baseline per service category)


,coverage_pct
service_name,
Korean restaurants,96.2
Coffee and beverages,94.3
Korean snack-food restaurants,78.7
Beer and casual pubs,78.3
Bakeries,69.4
Fried-chicken shops,67.7
Fast-food restaurants,64.6
Chinese restaurants,64.2
Western restaurants,60.3


Note: the lower a service category's coverage, the narrower the pool of commercial areas usable as controls.


## 1-7. Completeness check ②  Quarter continuity

Check whether each `(commercial area, service category)` combination has all 12 quarters. **Cells with fewer than 12 have a break in quarters midway, so some labels are not created in STEP 3.**

In [10]:
cnt = (store.groupby(["area_code", "service_code"])["quarter_code"]
       .nunique().rename("n_observed_quarters"))
dist = cnt.value_counts().sort_index(ascending=False)
display(pd.DataFrame({"n_cells": dist, "share": (dist/len(cnt)).round(3)}))
print(f"Of {len(cnt):,} (commercial area × service category) combinations, {(cnt==12).sum():,} have all 12 quarters ({(cnt==12).mean():.1%})")

# Continuity: are the observed quarters consecutive, with no gap in the middle?
qidx = {q: i for i, q in enumerate(QUARTERS)}
def is_gapless(s):
    v = sorted(qidx[q] for q in s)
    return v == list(range(v[0], v[0] + len(v)))
gap = store.groupby(["area_code", "service_code"])["quarter_code"].apply(is_gapless)
print(f"Cells with a gap in observed quarters : {(~gap).sum():,} ({(~gap).mean():.1%})")
print("→ these are what the STEP 3 'quarter continuity' check filters out")

,n_cells,share
n_observed_quarters,,
13,11522,0.881
12,141,0.011
11,147,0.011
10,150,0.011
9,140,0.011
8,120,0.009
7,126,0.010
6,131,0.010
5,142,0.011


Of 13,074 (commercial area × service category) combinations, 141 have all 12 quarters (1.1%)


Cells with a gap in observed quarters : 139 (1.1%)
→ these are what the STEP 3 'quarter continuity' check filters out


## 1-8. Sales — filter after de-duplication

The sales file for 2025 and the latest file without a year in its name overlap over 20251~20254. Keep the latest file's value by key.

In [11]:
sales_parts = []
for nm, raw in [("2023", sales23_raw), ("2024", sales24_raw),
                ("2025", sales25_raw), ("late", sales_late_raw)]:
    d = raw.copy()
    d = d[d["quarter_code"].isin(QUARTERS) & d["service_code"].isin(CODES)][SALES_KEEP]
    print(f"  {nm:5} {len(d):>7,} rows  quarters {sorted(d['quarter_code'].unique())}")
    sales_parts.append(d)

sales = pd.concat(sales_parts, ignore_index=True)
skey = ["quarter_code", "area_code", "service_code"]
before = len(sales)
sales = sales.drop_duplicates(subset=skey, keep="last").reset_index(drop=True)
print(f"\nDe-duplication {before:,} → {len(sales):,}  (-{before-len(sales):,})")
assert not sales.duplicated(skey).any(), "Duplicate keys remain"
print(f"Quarter coverage {sorted(sales['quarter_code'].unique())}")

  2023   27,430 rows  quarters [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234)]
  2024   27,095 rows  quarters [np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244)]
  2025   26,460 rows  quarters [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254)]
  late   33,033 rows  quarters [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]

De-duplication 114,018 → 87,558  (-26,460)
Quarter coverage [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]


## 1-9. Three commercial-area attribute tables

The area table does not change over time, so it has no quarter filter. The street-level population and attracting-facilities tables exist from 2021, so they are cut to 12 quarters.

These three are small (tens of thousands of rows), so all columns are kept — leaving room to use variables by sex, age group and facility type in sub-question 2.

In [12]:
AREA_KEEP = ["area_code", "area_type_name", "district_code", "district_name",
             "admin_dong_code", "area_size", "x_coord", "y_coord"]

area = area_raw.copy()[AREA_KEEP]
assert area["area_code"].is_unique, "Duplicate area_code in the area table"
print(f"area     {area.shape}  commercial areas {area['area_code'].nunique():,}")
display(area["area_type_name"].value_counts().to_frame("n_areas"))

flow = flow_raw.copy()
flow = flow[flow["quarter_code"].isin(QUARTERS)].drop(
    columns=["area_type_code", "area_type_name"])
print(f"flow     {flow.shape}  duplicate keys {flow.duplicated(['quarter_code','area_code']).sum()}")

facility = facility_raw.copy()
facility = facility[facility["quarter_code"].isin(QUARTERS)].drop(
    columns=["area_type_code", "area_type_name"])
print(f"facility {facility.shape}  duplicate keys {facility.duplicated(['quarter_code','area_code']).sum()}")
print(f"         commercial areas covered by attracting facilities {facility['area_code'].nunique():,} "
      f"(out of {area['area_code'].nunique():,} in the area table)")

area     (1650, 8)  commercial areas 1,650


,n_areas
area_type_name,
neighborhood commercial area,1090
traditional market,305
developed commercial area,249
tourist special zone,6


flow     (21434, 11)  duplicate keys 0
facility (20514, 4)  duplicate keys 0
         commercial areas covered by attracting facilities 1,578 (out of 1,650 in the area table)


## 1-10. Save

File size drops sharply compared with before (stores 166MB → a few MB).

In [13]:
out = {"01_store.pkl": store, "01_sales.pkl": sales,
       "01_area.pkl": area, "01_flow.pkl": flow, "01_facility.pkl": facility}
for nm, d in out.items():
    d.to_pickle(PROC / nm)
    mb = (PROC / nm).stat().st_size / 1024**2
    print(f"[saved] {nm:16} {str(d.shape):>16}   {mb:6.1f} MB")

print("\nThe next step, STEP 2, uses the following as is.")
print("  Total store-count column = 'store_count'  (not the earlier 'nonfranchise_store_count')")
print("  area_total_stores / area_food_stores / food_service_share / franchise_share are already computed in STEP 1")

[saved] 01_store.pkl         (160352, 14)     16.2 MB
[saved] 01_sales.pkl           (87558, 5)      4.0 MB
[saved] 01_area.pkl             (1650, 8)      0.1 MB
[saved] 01_flow.pkl           (21434, 11)      1.8 MB
[saved] 01_facility.pkl        (20514, 4)      0.6 MB

The next step, STEP 2, uses the following as is.
  Total store-count column = 'store_count'  (not the earlier 'nonfranchise_store_count')
  area_total_stores / area_food_stores / food_service_share / franchise_share are already computed in STEP 1


---

## What to fix in STEP 2

- The total store-count column name changed from `nonfranchise_store_count` → **`store_count`**
- Do not rebuild `area_total_stores` and `area_food_stores` in STEP 2 (they were built here)
- Keep the post-left-join row-count check logic as is